In [ ]:
import pandas as pd
import numpy as np

# 3. Tải dữ liệu (Nếu file cùng thư mục, chỉ cần để tên file)
# Nếu file nằm ở C:\Users\Admin\, hãy dùng đường dẫn r'C:\Users\Admin\dulieuxettuyendaihoc.csv'
try:
    df = pd.read_csv('dulieuxettuyendaihoc.csv')
except FileNotFoundError:
    df = pd.read_csv(r'C:\Users\Admin\dulieuxettuyendaihoc.csv')

print("10 dòng đầu:\n", df.head(10))
print("\n10 dòng cuối:\n", df.tail(10))

# 4. Xử lý thiếu cột DT
print("\nSố liệu thiếu cột DT:\n", df['DT'].isnull().value_counts())
print("Giá trị duy nhất của DT:", df['DT'].unique())
df['DT'] = df['DT'].fillna(0)

# 5 & 6. Xử lý thiếu cho T1 và các cột điểm khác bằng Mean
# Các cột không phải là điểm cần loại bỏ khỏi vòng lặp xử lý Mean
cols_to_exclude = ['STT', 'GT', 'DT', 'KV', 'KT']
cols_diem = [c for c in df.columns if c not in cols_to_exclude]

for col in cols_diem:
    df[col] = df[col].fillna(df[col].mean())

# 7. Tạo các biến TBM1, TBM2, TBM3
def calc_tbm(df, p):
    return (df[f'T{p}']*2 + df[f'L{p}'] + df[f'H{p}'] + df[f'S{p}'] + 
            df[f'V{p}']*2 + df[f'X{p}'] + df[f'D{p}'] + df[f'N{p}']) / 10

df['TBM1'] = calc_tbm(df, '1')
df['TBM2'] = calc_tbm(df, '2')
df['TBM3'] = calc_tbm(df, '6')

# 8. Tạo biến xếp loại XL1, XL2, XL3
def xep_loai(diem):
    if diem < 5.0: return 'Y'
    elif diem < 6.5: return 'TB'
    elif diem < 8.0: return 'K'
    elif diem < 9.0: return 'G'
    else: return 'XS'

for i in [1, 2, 3]:
    df[f'XL{i}'] = df[f'TBM{i}'].apply(xep_loai)

# 9. Chuyển đổi thang điểm 4 (Min-Max Normalization)
for i in [1, 2, 3]:
    col = f'TBM{i}'
    df[f'US_TBM{i}'] = (df[col] - df[col].min()) / (df[col].max() - df[col].min()) * 4

# 10. Tạo biến kết quả xét tuyển (KQXT)
def xet_tuyen(row):
    d1, d2, d3, kt = row['DH1'], row['DH2'], row['DH3'], row['KT']
    if kt in ['A', 'A1']:
        diem = (d1*2 + d2 + d3) / 4
    elif kt == 'B':
        diem = (d1 + d2*2 + d3) / 4
    else:
        diem = (d1 + d2 + d3) / 3
    return 1 if diem >= 5.0 else 0

df['KQXT'] = df.apply(xet_tuyen, axis=1)

# 11. Lưu file
df.to_csv('processed_dulieuxettuyendaihoc.csv', index=False)
print("\nĐã xử lý xong và lưu file: processed_dulieuxettuyendaihoc.csv")